# Python Basics & Exploratory Data Analysis (EDA) with Pandas

**Live class notebook — E-commerce dataset**

### What we cover
1. Python warm-up: calculating an average manually vs. with NumPy
2. Loading data with Pandas
3. Exploring the dataset (shape, columns, data types)
4. Fixing data types
5. Handling missing values
6. Filtering data
7. Finding and removing duplicates

---
## 1. Python Warm-up: Find the Average

We start with a simple list of numbers and compute the average in two ways.

In [ ]:
numbers = [12, 23, 45, 67, 98, 101, 23]

### 1.1 The manual way (using a loop)
Average = sum of all values ÷ number of values.

In [ ]:
number_add = 0   # running total
count = 0        # how many numbers we have seen

for num in numbers:
    number_add += num
    count += 1

print("Sum:", number_add)
print("Count:", count)
print("Average:", number_add / count)

### 1.2 The easy way (using NumPy)
Libraries like **NumPy** already have this built in — one line does the same job.

In [ ]:
import numpy as np

np.mean(numbers)

---
## 2. Libraries for Data Work

| Library | Used for |
|---|---|
| `numpy` | Numerical computing, fast math on arrays |
| `pandas` | Data manipulation — tables (DataFrames) |
| `dash` | Building interactive dashboards (used later in the course) |

`pandas` and `numpy` come pre-installed in Google Colab. `dash` has to be installed.

In [ ]:
# Install dash (only needed once per Colab session)
!pip install -q dash

In [ ]:
import pandas as pd
import dash

---
## 3. Load the Dataset

The dataset (`ecommerce_dataset_updated.xlsx`) is stored in Google Drive, so we first mount Drive in Colab.

> **Running locally (not in Colab)?** Skip the `drive.mount` cell and change `file_path` to where the file is on your computer.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
file_path = r'c:\Users\Firoj\OneDrive\Ostad\AI Engineer\Project 1\ecommerce_dataset_updated.xlsx'

# sheet_name=0 → read the first sheet of the Excel file
ecommerce_data = pd.read_excel(file_path, sheet_name=0)

---
## 4. Explore the Dataset

First, get to know the data: its size, columns, and what the rows look like.

In [ ]:
# (rows, columns)
ecommerce_data.shape

In [ ]:
# Column names
ecommerce_data.columns

In [ ]:
# First few rows
ecommerce_data.head()

In [ ]:
# Column data types and non-null counts
ecommerce_data.info()

### 4.1 Selecting specific columns
Pass a **list** of column names inside `[]` (hence the double brackets).

In [ ]:
ecommerce_data[['User_ID', 'Category']]

### 4.2 Unique values in a column

In [ ]:
ecommerce_data['Category'].unique()

---
## 5. Fix Data Types

In `info()` we saw that **`Price (Rs.)` is `object` (text)** even though it should be a number.
That happens when a column contains bad values — for example, the text `"ABC"` appears in some rows.

### 5.1 String → number
`pd.to_numeric(..., errors='coerce')` converts what it can and turns invalid values (like `"ABC"`) into `NaN` (missing).

In [ ]:
# Rows whose Price is present but NOT a valid number (e.g. "ABC")
bad_price = pd.to_numeric(ecommerce_data['Price (Rs.)'], errors='coerce').isna() & ecommerce_data['Price (Rs.)'].notnull()
ecommerce_data[bad_price]

In [ ]:
ecommerce_data['Price (Rs.)'] = pd.to_numeric(ecommerce_data['Price (Rs.)'], errors='coerce')

> ⚠️ **Careful with assignment!** Writing `ecommerce_data['Price (Rs.)'] = 8` would overwrite the **whole column** with the value 8. Assign a full Series (like above), not a single value, unless that is really what you want.

### 5.2 String → date

In [ ]:
ecommerce_data['Purchase_Date'] = pd.to_datetime(ecommerce_data['Purchase_Date'])

Check the data types again — `Price (Rs.)` should now be `float64` and `Purchase_Date` should be `datetime64`.

In [ ]:
ecommerce_data.info()

---
## 6. Handle Missing Values

### 6.1 Count missing values per column

In [ ]:
ecommerce_data.isna().sum()

### 6.2 View rows with / without missing values
- `.isna()` → `True` where the value is missing
- `.notnull()` → `True` where the value is present

In [ ]:
# Rows where Price is missing
ecommerce_data[ecommerce_data['Price (Rs.)'].isna()]

In [ ]:
# Rows where Price is present
ecommerce_data[ecommerce_data['Price (Rs.)'].notnull()]

### 6.3 Impute (fill) missing values
Common strategies for numeric columns:

| Strategy | When to use |
|---|---|
| **Mean** | Data is roughly symmetric, no big outliers |
| **Median** | Data is skewed or has outliers (safer default) |
| **Max / Min / constant** | Only when there is a business reason |

In [ ]:
# Price → fill with the mean
ecommerce_data['Price (Rs.)'] = ecommerce_data['Price (Rs.)'].fillna(ecommerce_data['Price (Rs.)'].mean())

# Discount → fill with the median
ecommerce_data['Discount (%)'] = ecommerce_data['Discount (%)'].fillna(ecommerce_data['Discount (%)'].median())

# Final price → fill with the max (demo of another strategy; median is usually the safer choice)
ecommerce_data['Final_Price(Rs.)'] = ecommerce_data['Final_Price(Rs.)'].fillna(ecommerce_data['Final_Price(Rs.)'].max())

Verify — these three columns should now show `0` missing values.

In [ ]:
ecommerce_data.isna().sum()

---
## 7. Filter Data

Syntax: `dataframe[condition]`

### 7.1 Single condition

In [ ]:
ecommerce_data_sports = ecommerce_data[ecommerce_data['Category'] == 'Sports']
ecommerce_data_sports

### 7.2 Multiple conditions
- Use `&` for **AND**, `|` for **OR**
- Wrap **each** condition in parentheses `( )`

In [ ]:
upi_high_value = ecommerce_data[(ecommerce_data['Payment_Method'] == 'UPI') &
                                (ecommerce_data['Final_Price(Rs.)'] > 400)]
upi_high_value

---
## 8. Find and Remove Duplicates

### 8.1 Find duplicate rows
`duplicated()` marks a row `True` if an identical row appeared earlier.

In [ ]:
ecommerce_data[ecommerce_data.duplicated()]

Confirm by looking at one of the duplicated users:

In [ ]:
ecommerce_data[ecommerce_data['User_ID'] == '337c166f']

### 8.2 Drop duplicates
`keep='first'` keeps the first occurrence and removes the rest.

In [ ]:
print("Before:", ecommerce_data.shape)

ecommerce_data = ecommerce_data.drop_duplicates(keep='first')

print("After: ", ecommerce_data.shape)

---
## Summary

| Step | Method |
|---|---|
| Load Excel data | `pd.read_excel()` |
| Explore | `.shape`, `.columns`, `.head()`, `.info()`, `.unique()` |
| Fix data types | `pd.to_numeric(errors='coerce')`, `pd.to_datetime()` |
| Missing values | `.isna().sum()`, `.fillna(mean / median / max)` |
| Filter | `df[condition]`, combine with `&` (AND) / `\|` (OR) |
| Duplicates | `.duplicated()`, `.drop_duplicates(keep='first')` |